# Movie Recommendation SystemA content-based recommender built on TMDB metadata.**Approach.** Each film is described by its genres, keywords, top-billed cast, and plotoverview. That text is vectorised with TF-IDF and films are compared by cosine similarity.Given a title, we return the nearest neighbours in that space.**Why not collaborative filtering?** Collaborative filtering learns from *user ratingpatterns* — "people who rated A highly also rated B highly". TMDB does not exposeper-user ratings through its public API, only aggregate vote averages, so a genuine CFmodel cannot be trained from this data source alone. Rather than ship a CF component thatsilently returns nothing, this notebook is honest about being content-based, and section 7shows how a real CF signal (fitted on the MovieLens ratings dataset) can be blended in.**Stack:** Python, requests, pandas, scikit-learn.> This product uses the TMDB API but is not endorsed or certified by TMDB.

## 1. SetupThe API key is read from the environment, never hard-coded — a key committed to git is aleaked key. Set it before launching Jupyter:```bashexport TMDB_API_KEY=your_key_here   # Windows: set TMDB_API_KEY=your_key_herejupyter notebook```

In [ ]:
import os
import json
import pathlib
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

API_KEY = os.environ.get("TMDB_API_KEY")
assert API_KEY, "TMDB_API_KEY is not set. See the cell above."

BASE = "https://api.themoviedb.org/3"
CACHE = pathlib.Path(".tmdb_cache")
CACHE.mkdir(exist_ok=True)

session = requests.Session()
pd.set_option("display.max_colwidth", 60)

## 2. Data acquisitionTwo endpoints are used:- `/movie/popular` — paginated list of popular films (20 per page)- `/movie/{id}` — full detail for one film, with `append_to_response` pulling keywords and  credits in the *same* HTTP call rather than three separate round tripsDetail responses are cached to disk as JSON. The first run costs ~500 requests; every runafter that costs zero. Requests are issued from a thread pool because they are IO-bound —serial fetching of 500 films takes minutes, concurrent fetching takes seconds.

In [ ]:
def get(path, **params):
    params.update(api_key=API_KEY, language="en-US")
    r = session.get(f"{BASE}{path}", params=params, timeout=15)
    r.raise_for_status()
    return r.json()


def fetch_popular(pages=25):
    rows = []
    for page in range(1, pages + 1):
        rows.extend(get("/movie/popular", page=page)["results"])
    return pd.DataFrame(rows).drop_duplicates(subset="id").reset_index(drop=True)


def fetch_details(movie_id):
    f = CACHE / f"{movie_id}.json"
    if f.exists():
        return json.loads(f.read_text())
    data = get(f"/movie/{movie_id}", append_to_response="keywords,credits")
    f.write_text(json.dumps(data))
    return data


movies = fetch_popular(pages=25)
with ThreadPoolExecutor(max_workers=8) as pool:
    movies["details"] = list(pool.map(fetch_details, movies["id"]))

print(f"{len(movies)} movies fetched")
movies[["id", "title", "release_date", "vote_average"]].head()

## 3. Feature engineeringThe text profile for each film combines four signals:| Signal | Why it helps ||---|---|| Genres | The coarsest and most reliable similarity axis || Keywords | Captures theme and tone that genre labels miss (*heist*, *dystopia*, *time_loop*) || Top 5 cast | Films sharing leads are often a meaningful recommendation || Overview | Free-text plot detail |Two details matter:- Multi-word names are joined with underscores, so `Christopher Nolan` becomes one token  rather than colliding with every other *Christopher* in the catalogue.- Genres are repeated three times. TF-IDF weights by frequency, and a single genre mention  would be drowned out by a 100-word overview.

In [ ]:
def genres_of(d):
    return " ".join(g["name"] for g in d.get("genres", []))


def keywords_of(d):
    kws = d.get("keywords", {}).get("keywords", [])
    return " ".join(k["name"].replace(" ", "_") for k in kws)


def cast_of(d, n=5):
    cast = d.get("credits", {}).get("cast", [])[:n]
    return " ".join(c["name"].replace(" ", "_") for c in cast)


def director_of(d):
    crew = d.get("credits", {}).get("crew", [])
    names = [c["name"].replace(" ", "_") for c in crew if c.get("job") == "Director"]
    return " ".join(names)


movies["genres"] = movies["details"].apply(genres_of)
movies["keywords"] = movies["details"].apply(keywords_of)
movies["cast"] = movies["details"].apply(cast_of)
movies["director"] = movies["details"].apply(director_of)
movies["overview"] = movies["overview"].fillna("")

movies["content"] = (
    (movies["genres"] + " ") * 3
    + movies["keywords"] + " "
    + movies["cast"] + " "
    + (movies["director"] + " ") * 2
    + movies["overview"]
)

movies[["title", "genres", "director"]].head()

## 4. Vectorise and compute similarity`min_df=2` drops tokens appearing in only one film — they can never link two films, so theyonly add noise and dimensions. Bigrams let phrases like *world war* survive as a unit.The diagonal of the similarity matrix is zeroed. Every film is perfectly similar to itself,and leaving that in means the top result is always the query. Zeroing it is cleaner thanslicing `[1:11]` and assuming the self-match is always first — that assumption breaks themoment two films tie.

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))
matrix = tfidf.fit_transform(movies["content"])

similarity = cosine_similarity(matrix, matrix)
np.fill_diagonal(similarity, 0.0)

print(f"TF-IDF matrix: {matrix.shape[0]} films x {matrix.shape[1]} features")
print(f"Sparsity: {100 * (1 - matrix.nnz / np.prod(matrix.shape)):.2f}% zeros")
print(f"Similarity matrix: {similarity.shape}")

## 5. RecommendationScores are returned *with* the recommendations. A recommender that hands back a bare listof titles is unauditable — you cannot tell a confident match from a coin flip.

In [ ]:
index_of = {mid: i for i, mid in enumerate(movies["id"])}
title_of = dict(zip(movies["id"], movies["title"]))


def recommend(movie_id, k=10):
    i = index_of.get(movie_id)
    if i is None:
        raise KeyError(f"movie_id {movie_id} not in catalogue")
    scores = similarity[i]
    top = np.argsort(scores)[::-1][:k]
    return pd.DataFrame({
        "title": movies.loc[top, "title"].values,
        "genres": movies.loc[top, "genres"].values,
        "score": scores[top].round(3),
    })


def recommend_by_title(title, k=10):
    hits = movies[movies["title"].str.lower() == title.lower()]
    if hits.empty:
        close = movies[movies["title"].str.contains(title, case=False, na=False)]
        raise KeyError(f"'{title}' not found. Did you mean: {list(close['title'][:5])}")
    return recommend(int(hits.iloc[0]["id"]), k)

In [ ]:
seed = movies.iloc[0]
print(f"Because you watched: {seed['title']}  ({seed['genres']})\n")
recommend(int(seed["id"]))

## 6. Does it actually work?An unevaluated recommender is a guess. There are no ground-truth labels here, so we use aproxy: **genre overlap**. If a recommendation shares no genre with the query film, the modelis probably wrong. Measured against a random baseline, this tells us whether the model haslearned anything at all.This is a weak metric — it rewards recommending *Action* films for *Action* films, which istrivial. It is a sanity check, not a benchmark. A real evaluation needs user data:precision@k against held-out watch history.

In [ ]:
def genre_set(i):
    return set(movies.at[i, "genres"].split())


def mean_genre_overlap(sample=100, k=10, random_baseline=False):
    rng = np.random.default_rng(42)
    idx = rng.choice(len(movies), size=min(sample, len(movies)), replace=False)
    scores = []
    for i in idx:
        gq = genre_set(i)
        if not gq:
            continue
        if random_baseline:
            recs = rng.choice(len(movies), size=k, replace=False)
        else:
            recs = np.argsort(similarity[i])[::-1][:k]
        overlap = [len(gq & genre_set(j)) / len(gq | genre_set(j) or {1}) for j in recs]
        scores.append(np.mean(overlap))
    return float(np.mean(scores))


model = mean_genre_overlap()
baseline = mean_genre_overlap(random_baseline=True)
print(f"Mean genre Jaccard, model:    {model:.3f}")
print(f"Mean genre Jaccard, random:   {baseline:.3f}")
print(f"Lift over random:             {model / baseline:.1f}x")

## 7. Adding a collaborative signal (optional)To make this a true hybrid, the CF half needs real user ratings. Download[MovieLens](https://grouplens.org/datasets/movielens/) (`ml-latest-small` is enough),fit item-item similarity on the ratings matrix, map MovieLens IDs to TMDB IDs via theincluded `links.csv`, and blend.The blend is a weighted sum of the two scores: `alpha` at 1.0 is pure content, 0.0 is pureCF. The right value is found empirically, not guessed.The cell below is scaffolding — it runs only if you have downloaded the dataset.

In [ ]:
def load_movielens_cf(path="ml-latest-small"):
    """Item-item cosine similarity over MovieLens ratings, keyed by TMDB id."""
    ratings = pd.read_csv(f"{path}/ratings.csv")
    links = pd.read_csv(f"{path}/links.csv")[["movieId", "tmdbId"]].dropna()
    links["tmdbId"] = links["tmdbId"].astype(int)

    pivot = ratings.pivot_table(index="movieId", columns="userId", values="rating").fillna(0)
    item_sim = cosine_similarity(pivot.values)
    np.fill_diagonal(item_sim, 0.0)

    ml_to_tmdb = dict(zip(links["movieId"], links["tmdbId"]))
    ml_ids = list(pivot.index)
    return item_sim, ml_ids, ml_to_tmdb


def blend(movie_id, cf_scores, alpha=0.6, k=10):
    """cf_scores: {tmdb_id: score}. alpha=1.0 pure content, 0.0 pure CF."""
    i = index_of[movie_id]
    content = {int(movies.at[j, "id"]): float(similarity[i][j]) for j in range(len(movies))}
    out = {}
    for mid in set(content) | set(cf_scores):
        if mid == movie_id:
            continue
        out[mid] = alpha * content.get(mid, 0.0) + (1 - alpha) * cf_scores.get(mid, 0.0)
    ranked = sorted(out.items(), key=lambda x: x[1], reverse=True)[:k]
    return pd.DataFrame(
        [(title_of.get(m, "?"), round(s, 3)) for m, s in ranked], columns=["title", "score"]
    )


# item_sim, ml_ids, ml_to_tmdb = load_movielens_cf()
# blend(seed_id, cf_scores_for(seed_id), alpha=0.6)

## 8. Persist the modelOnly what inference needs is saved: the dataframe minus the raw JSON blobs, the similaritymatrix as float32 (half the size, no meaningful precision loss for a ranking task), and thefitted vectoriser for scoring unseen text.

In [ ]:
import joblib

SLIM = ["id", "title", "genres", "director", "overview", "content", "release_date", "vote_average"]

joblib.dump(
    {"df": movies[SLIM], "sim": similarity.astype(np.float32), "tfidf": tfidf},
    "recommender.joblib",
    compress=3,
)

size_mb = pathlib.Path("recommender.joblib").stat().st_size / 1e6
print(f"Saved recommender.joblib ({size_mb:.1f} MB)")

## 9. LimitationsWorth stating plainly — a portfolio project that claims no weaknesses reads as one whoseauthor did not look for them.- **Catalogue is 500 popular films.** Recommendations can only come from this pool, which  skews recent and mainstream. Ask for something like *Solaris* and you will not get it.- **Content-based only.** No user data means no personalisation and no serendipity: the  model can only find films that *look alike on paper*, never "people like you also enjoyed".- **Cold start is inverted.** Unlike CF, new films are handled fine, but new *users* cannot  be modelled at all.- **Popularity is not quality.** The `/movie/popular` endpoint reflects current traffic,  so the catalogue drifts week to week.- **Evaluation is a proxy.** Genre overlap confirms the model is not random. It does not  confirm the recommendations are *good*.**Next steps:** swap `/movie/popular` for `/discover/movie` to build a larger,time-stratified catalogue; add the MovieLens CF signal from section 7; serve it behind aFastAPI endpoint.